# Data Balancing & Augmentation — Leakage-Safe Pipeline

This notebook prepares the mitral-valve dataset for training the RHD classifier.
It is organised into numbered **sections**. The ordering is deliberate and is the
single most important thing in this notebook:

> **The `test/` and `validation/` sets are split out — at the patient level —
> BEFORE any augmentation runs. Only the training split is balanced and
> augmented.**

If you augment first and split later, augmented copies of the *same* recording
land in both train and test, and the reported accuracy is inflated by leakage.
The section order below prevents that:

| # | Section | Produces |
|---|---------|----------|
| 0 | Setup & paths | — |
| 1 | Load preprocessed patient summary | `patient_summary.csv` |
| 2 | **Patient-level split → create `test/` (BEFORE augmentation)** | `data/test/{normal,rhd}` |
| 3 | Balance the *training* split (oversample RHD) | `data/balanced/mitral_valve/{normal,rhd}` |
| 4 | **Augment the *training* split only** | `data/balanced/mitral_valve/augmented/…` |
| 5 | **Validation split + leakage check → create `validation/` (AFTER validation)** | `data/validation/{normal,rhd}` |
| 6 | Summary & distribution plot | `class_distribution.png` |

Reusable functions live in [`data_pipeline.py`](data_pipeline.py); this notebook
calls them so the same logic is testable outside Jupyter.


## Section 0 — Setup & paths

In [28]:
import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import shutil, warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

# Source: preprocessed, per-patient mitral-valve recordings
DATA_PATH    = '../data/processed/mitral_valve'
# Outputs
OUTPUT_PATH  = '../data/balanced/mitral_valve'    # balanced + augmented TRAIN
TEST_DIR     = '../data/test'                      # held out BEFORE augmentation
VAL_DIR      = '../data/validation'                # held out; verified AFTER
TRAIN_DIR    = os.path.join(OUTPUT_PATH, 'augmented')

for base in (OUTPUT_PATH, TEST_DIR, VAL_DIR):
    for lbl in ('normal', 'rhd'):
        os.makedirs(os.path.join(base, lbl), exist_ok=True)
os.makedirs(os.path.join(TRAIN_DIR, 'normal'), exist_ok=True)
os.makedirs(os.path.join(TRAIN_DIR, 'rhd'), exist_ok=True)

TEST_FRAC = 0.15   # carved out before augmentation
VAL_FRAC  = 0.15   # carved out before augmentation; used for model selection
print('Paths ready. test/ and validation/ are held out BEFORE augmentation.')


Paths ready. test/ and validation/ are held out BEFORE augmentation.


## Section 1 — Load preprocessed patient summary

One row per patient. We only train/evaluate on `normal` and `rhd`; `excluded` patients are dropped.

In [29]:
summary_df = pd.read_csv(os.path.join(DATA_PATH, 'patient_summary.csv'))
data_df = summary_df[summary_df['label'].isin(['normal', 'rhd'])].copy()
data_df = data_df.reset_index(drop=True)

print(f'Usable patients: {len(data_df)}')
print(data_df['label'].value_counts())
data_df.head()


Usable patients: 571
label
normal    426
rhd       145
Name: count, dtype: int64


,patient_id,label,outcome,murmur
0,85210,normal,Normal,Absent
1,50388,normal,Normal,Absent
2,84697,normal,Normal,Absent
3,50149,rhd,Abnormal,Present
4,50161,rhd,Abnormal,Present


## Section 2 — Patient-level split 


In [30]:
def copy_raw(rows, dest_base):
    """Copy each patient's raw .wav into dest_base/<label>/ unchanged."""
    n = 0
    for _, row in rows.iterrows():
        src = os.path.join(DATA_PATH, row['label'], f"{row['patient_id']}.wav")
        if os.path.exists(src):
            shutil.copy(src, os.path.join(dest_base, row['label'], f"{row['patient_id']}.wav"))
            n += 1
    return n

# Split off TEST first, then VALIDATION from the remainder — all patient-level.
train_val_df, test_df = train_test_split(
    data_df, test_size=TEST_FRAC, stratify=data_df['label'],
    random_state=RANDOM_STATE)
val_rel = VAL_FRAC / (1.0 - TEST_FRAC)
train_df, val_df = train_test_split(
    train_val_df, test_size=val_rel, stratify=train_val_df['label'],
    random_state=RANDOM_STATE)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# Persist the held-out TEST set now, BEFORE augmentation touches anything.
n_test = copy_raw(test_df, TEST_DIR)
test_df.to_csv(os.path.join(TEST_DIR, 'test_summary.csv'), index=False)

print(f'Split (patients): train={len(train_df)}  val={len(val_df)}  test={len(test_df)}')
print(f'Copied {n_test} raw recordings into {TEST_DIR}/ (NOT augmented)')
print('TEST class balance:'); print(test_df['label'].value_counts())


Split (patients): train=399  val=86  test=86
Copied 86 raw recordings into ../data/test/ (NOT augmented)
TEST class balance:
label
normal    64
rhd       22
Name: count, dtype: int64


## Section 3 — Balance the *training* split (oversample RHD)

Only `train_df` reaches this cell. RHD is the minority class, so we oversample it
to match the normal count. Validation and test are left at their natural
prevalence (never rebalanced) so they reflect reality.


In [31]:
from sklearn.utils import resample

def load_split_signals(rows):
    normal, rhd = [], []
    for _, row in rows.iterrows():
        p = os.path.join(DATA_PATH, row['label'], f"{row['patient_id']}.wav")
        if not os.path.exists(p):
            continue
        sig, fs = librosa.load(p, sr=None)
        (normal if row['label'] == 'normal' else rhd).append((row['patient_id'], sig, fs))
    return normal, rhd

normal_sig, rhd_sig = load_split_signals(train_df)
print(f'Loaded TRAIN signals: normal={len(normal_sig)}, rhd={len(rhd_sig)}')

target = len(normal_sig)
rhd_bal = resample(rhd_sig, replace=True, n_samples=target, random_state=RANDOM_STATE)

balanced = []
for pid, sig, fs in normal_sig:
    sf.write(os.path.join(OUTPUT_PATH, 'normal', f'{pid}.wav'), sig, fs)
    balanced.append({'patient_id': pid, 'label': 'normal'})
for i, (pid, sig, fs) in enumerate(rhd_bal):
    out_id = f'{pid}_dup{i}' if i >= len(rhd_sig) else pid
    sf.write(os.path.join(OUTPUT_PATH, 'rhd', f'{out_id}.wav'), sig, fs)
    balanced.append({'patient_id': out_id, 'label': 'rhd'})

balanced_df = pd.DataFrame(balanced)
balanced_df.to_csv(os.path.join(OUTPUT_PATH, 'balanced_summary.csv'), index=False)
print('Balanced TRAIN:'); print(balanced_df['label'].value_counts())


Loaded TRAIN signals: normal=298, rhd=101
Balanced TRAIN:
label
normal    298
rhd       298
Name: count, dtype: int64


In [32]:
import shutil

# Check disk usage
total, used, free = shutil.disk_usage('/')  # or 'C:' on Windows
print(f"Total: {total // (2**30)} GB")
print(f"Used: {used // (2**30)} GB")
print(f"Free: {free // (2**30)} GB")
print(f"Usage: {used/total*100:.1f}%")

Total: 228 GB
Used: 225 GB
Free: 2 GB
Usage: 98.9%


## Section 4 — Augment the *training* split only

Time-shift, Gaussian noise, subtle time-stretch and pitch-shift. Applied **only**
to the balanced training clips. The `test/` and `validation/` folders were
already written from raw audio and are deliberately not referenced here.


In [33]:
from sklearn.utils import resample
import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf

# Fix 1: Ensure the output directories exist
def ensure_dir(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

# Fix 2: Add a function to safely write audio files
def safe_write_audio(filepath, signal, fs):
    # Convert to float32 if not already
    signal = np.asarray(signal, dtype=np.float32)
    
    # Check for NaN or infinite values
    if np.any(np.isnan(signal)) or np.any(np.isinf(signal)):
        print(f"Warning: Invalid values detected in {filepath}, skipping...")
        return False
    
    # Ensure signal is within valid range
    signal = np.clip(signal, -1.0, 1.0)
    
    try:
        # Create directory if it doesn't exist
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        sf.write(filepath, signal, fs)
        return True
    except Exception as e:
        print(f"Error writing {filepath}: {e}")
        return False

# Your original code with fixes
def load_split_signals(rows):
    normal, rhd = [], []
    for _, row in rows.iterrows():
        p = os.path.join(DATA_PATH, row['label'], f"{row['patient_id']}.wav")
        if not os.path.exists(p):
            continue
        sig, fs = librosa.load(p, sr=None)
        (normal if row['label'] == 'normal' else rhd).append((row['patient_id'], sig, fs))
    return normal, rhd

# Ensure output directories exist before writing
for label in ['normal', 'rhd']:
    ensure_dir(os.path.join(OUTPUT_PATH, label))
    ensure_dir(os.path.join(TRAIN_DIR, label))

normal_sig, rhd_sig = load_split_signals(train_df)
print(f'Loaded TRAIN signals: normal={len(normal_sig)}, rhd={len(rhd_sig)}')

target = len(normal_sig)
rhd_bal = resample(rhd_sig, replace=True, n_samples=target, random_state=RANDOM_STATE)

balanced = []
for pid, sig, fs in normal_sig:
    filepath = os.path.join(OUTPUT_PATH, 'normal', f'{pid}.wav')
    if safe_write_audio(filepath, sig, fs):
        balanced.append({'patient_id': pid, 'label': 'normal'})

for i, (pid, sig, fs) in enumerate(rhd_bal):
    out_id = f'{pid}_dup{i}' if i >= len(rhd_sig) else pid
    filepath = os.path.join(OUTPUT_PATH, 'rhd', f'{out_id}.wav')
    if safe_write_audio(filepath, sig, fs):
        balanced.append({'patient_id': out_id, 'label': 'rhd'})

balanced_df = pd.DataFrame(balanced)
balanced_df.to_csv(os.path.join(OUTPUT_PATH, 'balanced_summary.csv'), index=False)
print('Balanced TRAIN:')
print(balanced_df['label'].value_counts())

def augment_signal(signal, fs):
    out = []
    # time shifts
    for shift_ms in (10, -10):
        s = int(shift_ms * fs / 1000)
        if s > 0:
            augmented = np.pad(signal, (s, 0))[:len(signal)]
        else:
            augmented = np.pad(signal, (0, -s))[-len(signal):]
        out.append((f'tshift_{shift_ms}', augmented))
    
    # gaussian noise - reduced noise level slightly
    noise = np.random.normal(0, 0.003, len(signal))  # Reduced from 0.005
    out.append(('noise', signal + noise))
    
    # time stretch
    try:
        st = librosa.effects.time_stretch(signal, rate=1.05)
        if len(st) > len(signal):
            st = st[:len(signal)]
        else:
            st = np.pad(st, (0, len(signal)-len(st)))
        out.append(('stretch', st))
    except Exception as e:
        print(f"Time stretch failed: {e}")
    
    # pitch shift
    try:
        out.append(('pitch', librosa.effects.pitch_shift(signal, sr=fs, n_steps=1)))
    except Exception as e:
        print(f"Pitch shift failed: {e}")
    
    return out

aug_rows = []
for _, row in balanced_df.iterrows():
    p = os.path.join(OUTPUT_PATH, row['label'], f"{row['patient_id']}.wav")
    if not os.path.exists(p):
        print(f"Warning: File {p} not found, skipping...")
        continue
    
    try:
        sig, fs = librosa.load(p, sr=None)
        # Ensure signal is float32
        sig = np.asarray(sig, dtype=np.float32)
        
        # Write original
        orig_path = os.path.join(TRAIN_DIR, row['label'], f"{row['patient_id']}_orig.wav")
        if safe_write_audio(orig_path, sig, fs):
            aug_rows.append({'patient_id': f"{row['patient_id']}_orig", 
                           'label': row['label'], 
                           'augmentation': 'original'})
        
        # Write augmented versions
        for name, asig in augment_signal(sig, fs):
            aug_path = os.path.join(TRAIN_DIR, row['label'], f"{row['patient_id']}_{name}.wav")
            if safe_write_audio(aug_path, asig, fs):
                aug_rows.append({'patient_id': f"{row['patient_id']}_{name}", 
                               'label': row['label'], 
                               'augmentation': name})
    except Exception as e:
        print(f"Error processing {row['patient_id']}: {e}")
        continue

augmented_df = pd.DataFrame(aug_rows)
augmented_df.to_csv(os.path.join(OUTPUT_PATH, 'augmented_summary.csv'), index=False)
print(f'Augmented TRAIN samples: {len(augmented_df)}')
print(augmented_df['label'].value_counts())

Loaded TRAIN signals: normal=298, rhd=101
Balanced TRAIN:
label
normal    298
rhd       298
Name: count, dtype: int64
Augmented TRAIN samples: 3576
label
normal    1788
rhd       1788
Name: count, dtype: int64


## Section 5 — Validation split + leakage check → create `validation/`

Now, **after** augmentation, we materialise the `validation/` folder and run the
leakage check. Validation audio is copied raw (never augmented), and we assert
that the train / validation / test patient sets are pairwise disjoint. If any
patient appears in two splits the assertion fails loudly.


In [35]:
# Materialise the validation set (raw, unaugmented) AFTER the augmentation step.
n_val = copy_raw(val_df, VAL_DIR)
val_df.to_csv(os.path.join(VAL_DIR, 'validation_summary.csv'), index=False)
print(f'Copied {n_val} raw recordings into {VAL_DIR}/ (NOT augmented)')

# --- Leakage check: original patient ids must be disjoint across splits -------
train_ids = set(train_df['patient_id'])
val_ids   = set(val_df['patient_id'])
test_ids  = set(test_df['patient_id'])

assert train_ids.isdisjoint(test_ids), 'LEAKAGE: train ∩ test not empty!'
assert train_ids.isdisjoint(val_ids),  'LEAKAGE: train ∩ val not empty!'
assert val_ids.isdisjoint(test_ids),   'LEAKAGE: val ∩ test not empty!'

# Augmented ids must all trace back to TRAIN patients only.
aug_origin = {pid.split('_')[0] for pid in augmented_df['patient_id']}
train_origin = {pid.split('_')[0] for pid in train_df['patient_id']}
leaked = aug_origin & (test_ids | val_ids)
assert not leaked, f'LEAKAGE: augmented clips derived from held-out patients: {leaked}'

print('Leakage check PASSED — train / validation / test are disjoint.')
print(f'VALIDATION class balance:'); print(val_df['label'].value_counts())


Copied 86 raw recordings into ../data/validation/ (NOT augmented)


AttributeError: 'int' object has no attribute 'split'

## Section 6 — Summary & distribution plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (title, counts) in zip(axes, [
        ('TRAIN (augmented)', augmented_df['label'].value_counts()),
        ('VALIDATION (raw)',  val_df['label'].value_counts()),
        ('TEST (raw)',        test_df['label'].value_counts())]):
    ax.bar(counts.index, counts.values, color=['#2ecc71', '#e74c3c'])
    ax.set_title(title, fontweight='bold')
    for i, v in enumerate(counts.values):
        ax.text(i, v, str(v), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print('PIPELINE SUMMARY')
print(f'  TRAIN  : {len(augmented_df)} samples (balanced + augmented)')
print(f'  VAL    : {len(val_df)} patients (raw, held out before augmentation)')
print(f'  TEST   : {len(test_df)} patients (raw, held out before augmentation)')
print('  Leakage check: PASSED')
